# NB07 — Reporting and Figures
**MSK | Goel Lab** · LLM Systematic Review · Method Comparison

**Purpose**: Generate all manuscript figures and tables from upstream analysis outputs.

**Figures produced**
- Fig 1: PRISMA flow diagram (Boolean search)
- Fig 2: Precision / Recall / F1 grouped bar chart (3 methods)
- Fig 3: NIAH heatmap — Standard RAG (depth × context length)
- Fig 4: DeepEval radar chart — 5 RAG metrics (Standard vs Agentic)
- Fig 5: Thematic coverage heatmap (3 methods × themes)
- Fig 6: Time-to-results + screening burden comparison

**Outputs** → `reports/figures/`

In [ ]:
import sys
sys.path.insert(0, '..')

from src.utils.io import load_env, data_dir, project_root
load_env()

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

FIG_DIR = project_root() / 'reports' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Plot style
sns.set_theme(style='whitegrid', palette='muted')
COLORS = {'boolean': '#2196F3', 'standard_rag': '#4CAF50', 'agentic_rag': '#FF9800'}

## Fig 2 — Precision / Recall / F1 grouped bar chart

In [ ]:
comp_df = pd.read_csv(data_dir() / 'processed' / 'method_comparison.csv')

metrics = ['precision', 'recall', 'f1']
x = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(8, 5))
for i, (_, row) in enumerate(comp_df.iterrows()):
    method = row['method']
    vals = [row[m] for m in metrics]
    ax.bar(x + i * width, vals, width, label=method.replace('_', ' ').title(),
           color=COLORS.get(method, '#999'))

ax.set_xticks(x + width)
ax.set_xticklabels(['Precision', 'Recall', 'F1'])
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Retrieval Performance by Method')
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig2_precision_recall_f1.pdf', dpi=300)
fig.savefig(FIG_DIR / 'fig2_precision_recall_f1.png', dpi=300)
plt.show()
print('Fig 2 saved.')

## Fig 3 — NIAH heatmap (Standard RAG)

In [ ]:
niah_path = project_root() / 'reports' / 'niah_results_standard_rag.json'

if niah_path.exists():
    niah = json.loads(niah_path.read_text())
    ctx_lengths = sorted(set(v['context_length'] for v in niah.values()))
    depths = sorted(set(v['depth'] for v in niah.values()))

    matrix = np.zeros((len(ctx_lengths), len(depths)))
    for v in niah.values():
        i = ctx_lengths.index(v['context_length'])
        j = depths.index(v['depth'])
        matrix[i, j] = 1 if (v['needle_retrieved'] and v['needle_in_answer']) else 0

    fig, ax = plt.subplots(figsize=(7, 4))
    sns.heatmap(matrix, annot=True, fmt='.0f', cmap='RdYlGn', vmin=0, vmax=1,
                xticklabels=[f'{int(d*100)}%' for d in depths],
                yticklabels=ctx_lengths, ax=ax, cbar_kws={'label': 'Pass (1) / Fail (0)'})
    ax.set_xlabel('Depth')
    ax.set_ylabel('Context length (tokens approx.)')
    ax.set_title('NIAH — Standard RAG')
    fig.tight_layout()
    fig.savefig(FIG_DIR / 'fig3_niah_standard_rag.pdf', dpi=300)
    fig.savefig(FIG_DIR / 'fig3_niah_standard_rag.png', dpi=300)
    plt.show()
    print('Fig 3 saved.')
else:
    print('NIAH results not yet available — run NB03 first.')

## Fig 6 — Time-to-results + screening burden

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, metric, label in zip(
    axes,
    ['time_seconds', 'screening_burden'],
    ['Time to results (s)', 'Screening burden (records)']
):
    sub = comp_df.dropna(subset=[metric])
    colors = [COLORS.get(m, '#999') for m in sub['method']]
    ax.barh(sub['method'].str.replace('_', ' ').str.title(), sub[metric], color=colors)
    ax.set_xlabel(label)
    ax.set_title(label)

fig.tight_layout()
fig.savefig(FIG_DIR / 'fig6_operational_metrics.pdf', dpi=300)
fig.savefig(FIG_DIR / 'fig6_operational_metrics.png', dpi=300)
plt.show()
print('Fig 6 saved.')